## LangGraph Durable Execution — StateGraph 기준

Durable Execution은 **그래프 실행 중 각 슈퍼스텝(superstep) 경계마다 체크포인트를 저장**하여, 중단·실패·사람 개입 후 정확히 같은 위치에서 재개할 수 있게 합니다.

이 튜토리얼에서는 `StateGraph`를 기준으로 각 개념을 완성 예제로 이해하고, 과제를 통해 직접 구현합니다.

| 섹션 | 개념 | 핵심 패턴 |
|------|------|----------|
| 2 | 기본 체크포인팅 | `compile(checkpointer=)`, `get_state()`, `get_state_history()` |
| 3 | 결함 복구 | 노드 실패 후 `invoke(None, config)` |
| 4 | Human-in-the-loop | `interrupt()` + `Command(resume=...)` |
| 5 | Time Travel | `get_state_history()` + `checkpoint_id`로 재실행 |
| 6 | 종합 과제 | 문서 처리 파이프라인 |

### 1. 환경 설정

StateGraph 기반 Durable Execution에 필요한 패키지와 핵심 모듈을 준비합니다.

```
TypedDict로 State 정의
  ↓
StateGraph(State)로 그래프 빌더 생성
  ↓
builder.compile(checkpointer=InMemorySaver())
  ↓
graph.invoke(input, {"configurable": {"thread_id": "..."}})
```

> **슈퍼스텝(superstep)**: 한 번의 스케줄링 사이클에서 실행되는 모든 노드의 묶음. 슈퍼스텝 완료 시마다 체크포인트가 저장된다.

In [ ]:
%pip install -qU langgraph langchain-anthropic

In [12]:
import random
import time
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

print("환경 설정 완료")

환경 설정 완료


### 2. 기본 체크포인팅 — `get_state` / `get_state_history`

**체크포인터(Checkpointer)** 를 연결한 그래프는 `thread_id` 단위로 실행 히스토리를 저장합니다. 언제든 과거·현재 상태를 조회할 수 있습니다.

```
invoke → [START] → node_A → node_B → node_C → [END]
                  ↑체크포인트  ↑체크포인트  ↑체크포인트
```

| 메서드 | 반환 | 설명 |
|--------|------|------|
| `get_state(config)` | `StateSnapshot` | 현재(최신) 상태 |
| `get_state_history(config)` | `Iterator[StateSnapshot]` | 전체 체크포인트 히스토리 |

> **`StateSnapshot.next`**: 다음에 실행될 노드 이름 튜플. 빈 튜플 `()`이면 실행 완료.

In [13]:
# 기본 체크포인팅 예제: 3단계 리뷰 파이프라인

class ReviewState(TypedDict):
    topic: str
    data: str
    summary: str
    score: int

def fetch_data(state: ReviewState) -> dict:
    """1단계: 주제 관련 데이터를 수집한다."""
    print(f"  [fetch_data] topic='{state['topic']}'")
    return {"data": f"{state['topic']}에 대한 원본 데이터"}

def summarize(state: ReviewState) -> dict:
    """2단계: 데이터를 요약한다."""
    print(f"  [summarize] data='{state['data']}'")
    return {"summary": f"요약: {state['data'][:10]}..."}

def score_content(state: ReviewState) -> dict:
    """3단계: 요약 품질을 평가한다."""
    print(f"  [score_content] summary='{state['summary']}'")
    return {"score": random.randint(70, 100)}

review_builder = StateGraph(ReviewState)
review_builder.add_node("fetch_data", fetch_data)
review_builder.add_node("summarize", summarize)
review_builder.add_node("score_content", score_content)
review_builder.add_edge(START, "fetch_data")
review_builder.add_edge("fetch_data", "summarize")
review_builder.add_edge("summarize", "score_content")
review_builder.add_edge("score_content", END)

review_graph = review_builder.compile(checkpointer=InMemorySaver())

In [14]:
# 실행 후 get_state / get_state_history 확인
config = {"configurable": {"thread_id": "review-001"}}

review_graph.invoke({"topic": "LangGraph"}, config)

print("\n=== 현재 상태 (get_state) ===")
snapshot = review_graph.get_state(config)
print(f"  values : {snapshot.values}")
print(f"  next   : {snapshot.next}  ← 빈 튜플이면 실행 완료")
print(f"  step   : {snapshot.metadata.get('step')}")

print("\n=== 체크포인트 히스토리 (get_state_history) ===")
for i, s in enumerate(review_graph.get_state_history(config)):
    print(f"  [{i}] step={s.metadata.get('step'):>2}  next={s.next}")

  [fetch_data] topic='LangGraph'
  [summarize] data='LangGraph에 대한 원본 데이터'
  [score_content] summary='요약: LangGraph에...'

=== 현재 상태 (get_state) ===
  values : {'topic': 'LangGraph', 'data': 'LangGraph에 대한 원본 데이터', 'summary': '요약: LangGraph에...', 'score': 93}
  next   : ()  ← 빈 튜플이면 실행 완료
  step   : 3

=== 체크포인트 히스토리 (get_state_history) ===
  [0] step= 3  next=()
  [1] step= 2  next=('score_content',)
  [2] step= 1  next=('summarize',)
  [3] step= 0  next=('fetch_data',)
  [4] step=-1  next=('__start__',)


#### 🎯 과제 1: 점수 집계 파이프라인

3개 과목 점수를 단계적으로 처리하는 파이프라인을 구현하고, 실행 후 체크포인트 히스토리를 출력하세요.

| 노드 | 역할 | 반환 키 |
|------|------|--------|
| `load_scores` | 과목별 점수 dict 생성 | `scores` |
| `calc_average` | 평균 계산 | `average` |
| `grade` | 평균 → 등급 문자열 (A/B/C/F) | `grade` |

힌트: `get_state_history()` 결과를 순회하며 각 체크포인트의 `step`, `next`, `values`를 출력해보세요.

In [15]:
class ScoreState(TypedDict):
    student: str
    scores: dict
    average: float
    grade: str

def load_scores(state: ScoreState) -> dict:
    """과목별 점수를 로드한다."""
    return {"scores": {"math": random.randint(60, 100), "english": random.randint(60, 100), "science": random.randint(60, 100)}}

def calc_average(state: ScoreState) -> dict:
    """평균 점수를 계산한다."""
    return {"average": sum(state["scores"].values()) / len(state["scores"])}

def grade(state: ScoreState) -> dict:
    """평균 점수로 등급을 매긴다."""
    avg = state["average"]
    return {"grade": "A" if avg >= 90 else "B" if avg >= 80 else "C" if avg >= 70 else "F"}

# TODO [핵심]: checkpointer 없이 compile하면 get_state()가 동작하지 않는다.
#              반드시 compile(checkpointer=InMemorySaver()) 로 빌드할 것.
score_builder = StateGraph(ScoreState)
# TODO: add_node / add_edge 로 load_scores → calc_average → grade 연결
score_builder.add_node("load_scores", load_scores)
score_builder.add_node("calc_average", calc_average)
score_builder.add_node("grade", grade)
score_builder.add_edge(START, "load_scores")
score_builder.add_edge("load_scores", "calc_average")
score_builder.add_edge("calc_average", "grade")
score_builder.add_edge("grade", END)
score_graph = score_builder.compile(checkpointer=InMemorySaver())


config = {"configurable": {"thread_id": "score-001"}}
score_graph.invoke({"student": "홍길동"}, config)

# TODO [개념 확인]: get_state()의 .next 필드를 출력해보세요.
#                  실행이 완료된 경우 next는 어떤 값인가요?
snapshot = score_graph.get_state(config)
print(f"next = {snapshot.next}")   # 예상값: ()

# TODO [개념 확인]: get_state_history()는 최신→과거 순으로 반환됩니다.
#                  각 체크포인트의 step 번호와 next를 출력하여
#                  "노드 실행 전 체크포인트 저장" 패턴을 확인하세요.
for s in score_graph.get_state_history(config):
    print(f"step={s.metadata.get('step')}  next={s.next}  grade={s.values.get('grade')!r}")

next = ()
step=3  next=()  grade='A'
step=2  next=('grade',)  grade=None
step=1  next=('calc_average',)  grade=None
step=0  next=('load_scores',)  grade=None
step=-1  next=('__start__',)  grade=None


### 3. 결함 복구 — 노드 실패 후 재개

노드 실행 중 예외가 발생하면, 체크포인트는 **실패한 노드 직전 상태**로 저장됩니다. `invoke(None, config)` 로 재개하면 이미 완료된 노드는 건너뜁니다.

```
첫 실행:
  node_A 완료 → 체크포인트 저장
  node_B 실행 중 → 예외!
  → snapshot.next = ("node_B",)  ← 여기서 멈춤

invoke(None, config) 재개:
  node_A → 건너뜀 (체크포인트 로드)
  node_B → 재실행 → 성공
  node_C → 실행
```

> **`snapshot.next`**: 실패 후 `get_state(config).next`로 어느 노드에서 멈췄는지 확인할 수 있다.

In [16]:
# 결함 복구 예제: call_api 노드가 첫 번째 시도에서 실패

class PipelineState(TypedDict):
    raw: str
    cleaned: str
    result: str

api_call_count = {"count": 0}

def clean_data(state: PipelineState) -> dict:
    """1단계: 데이터 정제 — 항상 성공."""
    print("  [clean_data] 실행")
    return {"cleaned": state["raw"].strip().upper()}

def call_api(state: PipelineState) -> dict:
    """2단계: 외부 API 호출 — 첫 번째 시도 실패."""
    api_call_count["count"] += 1
    print(f"  [call_api] 시도 {api_call_count['count']}회")
    if api_call_count["count"] == 1:
        raise ConnectionError("API 서버 오류 (시뮬레이션)")
    return {"result": f"{state['cleaned']}_처리완료"}

def save_result(state: PipelineState) -> dict:
    """3단계: 결과 저장 — 항상 성공."""
    print("  [save_result] 실행")
    return {}

pipeline_builder = StateGraph(PipelineState)
pipeline_builder.add_node("clean_data", clean_data)
pipeline_builder.add_node("call_api", call_api)
pipeline_builder.add_node("save_result", save_result)
pipeline_builder.add_edge(START, "clean_data")
pipeline_builder.add_edge("clean_data", "call_api")
pipeline_builder.add_edge("call_api", "save_result")
pipeline_builder.add_edge("save_result", END)

pipeline_graph = pipeline_builder.compile(checkpointer=InMemorySaver())

In [ ]:
# 첫 번째 실행 — call_api에서 실패
config = {"configurable": {"thread_id": "pipeline-001"}}

print("=== 첫 번째 실행 ===")
try:
    pipeline_graph.invoke({"raw": "  hello world  "}, config)
except Exception as e:
    print(f"→ 예외: {e}")

snapshot = pipeline_graph.get_state(config)
print(f"→ 멈춘 위치 : {snapshot.next}")
print(f"→ 저장된 상태: cleaned='{snapshot.values.get('cleaned')}'")

In [ ]:
# invoke(None, config) 으로 재개 — clean_data 재실행 없음
print("=== invoke(None, config) 재개 ===")
print("(clean_data 로그가 없으면 체크포인트에서 로드된 것)\n")

result = pipeline_graph.invoke(None, config)
print(f"\n→ 최종 결과: {result}")

#### 🎯 과제 2: 3회 재시도 파이프라인

이미지 업로드 파이프라인을 구현하세요. `upload` 노드는 처음 2번 실패하고 3번째에 성공합니다. 루프를 사용해 성공할 때까지 재시도하세요.

| 노드 | 동작 |
|------|------|
| `resize` | 항상 성공 — `resized: "원본_resized"` |
| `upload` | 처음 2번 실패, 3번째 성공 |

힌트: 재시도 루프에서 실패 시 `input_val = None` 으로 바꿔 `invoke(None, config)` 로 재개하세요.

In [ ]:
class ImageState(TypedDict):
    original: str
    resized: str
    uploaded: str

upload_attempts = {"count": 0}

def resize(state: ImageState) -> dict:
    """이미지 크기를 조정한다."""
    print("  [resize] 실행")
    return {"resized": state["original"] + "_resized"}

def upload(state: ImageState) -> dict:
    """이미지를 업로드한다 — 처음 2번 실패."""
    upload_attempts["count"] += 1
    print(f"  [upload] 시도 {upload_attempts['count']}회")
    if upload_attempts["count"] <= 2:
        raise RuntimeError(f"업로드 실패 (시도 {upload_attempts['count']}회)")
    return {"uploaded": state["resized"] + "_uploaded"}

image_builder = StateGraph(ImageState)
image_builder.add_node("resize", resize)
image_builder.add_node("upload", upload)
image_builder.add_edge(START, "resize")
image_builder.add_edge("resize", "upload")
image_builder.add_edge("upload", END)
image_graph = image_builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "image-001"}}
input_val = {"original": "photo.jpg"}

for attempt in range(1, 5):
    try:
        result = image_graph.invoke(input_val, config)
        print(f"성공: {result}")
        break
    except Exception as e:
        print(f"시도 {attempt} 실패: {e}")

        # TODO [결함 복구 핵심]: 재시도 시 input_val을 None으로 바꿔야 하는 이유는?
        #   - None이 아닌 원래 입력을 넣으면: 체크포인트를 무시하고 처음부터 재실행됨
        #   - None을 넣으면: 마지막 체크포인트(resize 완료 후)에서 이어서 실행됨
        #   → resize가 재실행되지 않는다는 것을 로그로 확인하세요.
        input_val = None  # TODO: 이 줄을 주석 처리하면 어떻게 되는지 실험해보세요

# TODO [개념 확인]: 성공 후 get_state_history()를 출력하여
#   몇 개의 체크포인트가 저장됐는지 확인하세요.
#   (시도 횟수만큼 "upload 직전" 체크포인트가 쌓임)
for s in image_graph.get_state_history(config):
    print(f"  step={s.metadata.get('step')}  next={s.next}")

### 4. Human-in-the-loop — `interrupt` + `Command`

노드 내부에서 **`interrupt(값)`** 을 호출하면 그래프가 일시 중단됩니다. 사람이 확인 후 **`Command(resume=값)`** 으로 재개하면, `interrupt()`가 그 값을 반환합니다.

```
graph.stream(input, config)
  ↓
review_node 실행
  → interrupt({"draft": ...})  ← 중단, 이벤트에 __interrupt__ 발생
  ↓ (사람 응답)
graph.stream(Command(resume="승인"), config)
  → interrupt()의 반환값 = "승인"
  → 이후 로직 계속 실행
```

> **`update_state(config, values)`**: interrupt 전에 상태를 직접 수정해야 할 때 사용한다.

In [ ]:
# Human-in-the-loop 예제: 이메일 초안 검토 워크플로우

class EmailState(TypedDict):
    recipient: str
    topic: str
    draft: str
    feedback: str
    status: str

def write_draft(state: EmailState) -> dict:
    """이메일 초안을 작성한다."""
    draft = f"수신: {state['recipient']}\n제목: {state['topic']}\n내용: 자동 생성 이메일입니다."
    print("  [write_draft] 초안 작성 완료")
    return {"draft": draft}

def human_review(state: EmailState) -> dict:
    """사람이 초안을 검토한다."""
    feedback = interrupt({
        "message": "이메일 초안을 검토하세요.",
        "draft": state["draft"]
    })
    return {"feedback": feedback}

def send_email(state: EmailState) -> dict:
    """피드백에 따라 이메일을 발송하거나 취소한다."""
    if state["feedback"] == "승인":
        print("  [send_email] 발송 완료")
        return {"status": "발송 완료"}
    else:
        print(f"  [send_email] 취소 — 이유: {state['feedback']}")
        return {"status": f"취소: {state['feedback']}"}

email_builder = StateGraph(EmailState)
email_builder.add_node("write_draft", write_draft)
email_builder.add_node("human_review", human_review)
email_builder.add_node("send_email", send_email)
email_builder.add_edge(START, "write_draft")
email_builder.add_edge("write_draft", "human_review")
email_builder.add_edge("human_review", "send_email")
email_builder.add_edge("send_email", END)

email_graph = email_builder.compile(checkpointer=InMemorySaver())

In [ ]:
# 워크플로우 시작 — human_review에서 중단
config = {"configurable": {"thread_id": "email-001"}}
initial = {"recipient": "manager@company.com", "topic": "월간 보고서"}

print("=== 워크플로우 시작 ===")
for event in email_graph.stream(initial, config):
    if "__interrupt__" in event:
        data = event["__interrupt__"][0].value
        print(f"\n[중단] {data['message']}")
        print(f"\n--- 초안 ---\n{data['draft']}")
        print("\n→ 재개: graph.stream(Command(resume='승인'), config)")

In [ ]:
# 사람이 승인 → Command(resume='승인')으로 재개
print("=== 승인 후 재개 ===")
for event in email_graph.stream(Command(resume="승인"), config):
    if "send_email" in event:
        print(f"→ 최종 상태: {event['send_email']}")

In [ ]:
# 비교: 다른 thread_id로 거절 시나리오
config2 = {"configurable": {"thread_id": "email-002"}}

print("=== 거절 시나리오 ===")
for event in email_graph.stream(initial, config2):
    if "__interrupt__" in event:
        print("[중단] 검토 대기 중...")

for event in email_graph.stream(Command(resume="내용 수정 필요"), config2):
    if "send_email" in event:
        print(f"→ 최종 상태: {event['send_email']}")

#### 🎯 과제 3: 2단계 승인 워크플로우

구매 요청 워크플로우를 구현하세요. 금액에 따라 승인 단계가 달라집니다.

| 조건 | 흐름 |
|------|------|
| 금액 ≤ 100만원 | `create_request` → `manager_review` → `process` |
| 금액 > 100만원 | `create_request` → `manager_review` → `exec_review` → `process` |

- `manager_review`: `interrupt()`로 팀장 승인 요청. 거절 시 바로 `process`로 이동
- `exec_review`: `interrupt()`로 임원 승인 요청
- `process`: `manager_approved` / `exec_approved` 에 따라 `status` 결정

힌트: `add_conditional_edges("manager_review", route_fn, {"exec_review": ..., "process": ...})` 로 분기하세요.

In [ ]:
class PurchaseState(TypedDict):
    item: str
    amount: int
    request_id: str
    manager_approved: bool
    exec_approved: bool
    status: str

def create_request(state: PurchaseState) -> dict:
    """구매 요청서를 생성한다."""
    print(f"  [create_request] {state['item']} / {state['amount']:,}원")
    return {"request_id": f"PR-{random.randint(1000, 9999)}"}

def manager_review(state: PurchaseState) -> dict:
    """팀장 승인을 요청한다."""
    # TODO [interrupt 핵심]: interrupt()는 그래프를 일시 중단하고 값을 반환받는다.
    #   반환받은 값(True/False)을 manager_approved 에 저장한다.
    #   → approved = interrupt({"message": "팀장 승인 요청", "item": ..., "amount": ...})
    approved = interrupt({"message": "팀장 승인 요청", "item": state["item"], "amount": state["amount"]})
    return {"manager_approved": approved}

def exec_review(state: PurchaseState) -> dict:
    """임원 승인을 요청한다."""
    # TODO [interrupt 핵심]: 두 번째 interrupt — 같은 thread_id에서 순서대로 재개된다.
    #   첫 번째 Command(resume=...)가 manager_review를 재개,
    #   두 번째 Command(resume=...)가 이 interrupt를 재개한다.
    approved = interrupt({"message": "임원 최종 승인 요청", "item": state["item"], "amount": state["amount"]})
    return {"exec_approved": approved}

def process(state: PurchaseState) -> dict:
    """최종 처리 — 승인 여부에 따라 status를 결정한다."""
    if not state.get("manager_approved"):
        status = "팀장 거절"
    elif state["amount"] > 1_000_000 and not state.get("exec_approved"):
        status = "임원 거절"
    else:
        status = f"구매 완료: {state['item']} ({state['amount']:,}원)"
    print(f"  [process] {status}")
    return {"status": status}

def route_after_manager(state: PurchaseState) -> str:
    # TODO [조건부 엣지 핵심]: interrupt() 반환값(manager_approved)에 따라 다음 노드를 결정한다.
    #   - 거절(False) → "process" (즉시 종료)
    #   - 승인 + 고액   → "exec_review" (임원 승인 추가)
    #   - 승인 + 소액   → "process" (바로 처리)
    if not state["manager_approved"]:
        return "process"
    return "exec_review" if state["amount"] > 1_000_000 else "process"

purchase_builder = StateGraph(PurchaseState)
purchase_builder.add_node("create_request", create_request)
purchase_builder.add_node("manager_review", manager_review)
purchase_builder.add_node("exec_review", exec_review)
purchase_builder.add_node("process", process)
purchase_builder.add_edge(START, "create_request")
purchase_builder.add_edge("create_request", "manager_review")
# TODO [조건부 엣지]: add_conditional_edges로 route_after_manager 연결
#   manager_review → (route_after_manager) → exec_review 또는 process
purchase_builder.add_conditional_edges("manager_review", route_after_manager,
                                       {"exec_review": "exec_review", "process": "process"})
purchase_builder.add_edge("exec_review", "process")
purchase_builder.add_edge("process", END)
purchase_graph = purchase_builder.compile(checkpointer=InMemorySaver())

# ── 소액(50만) 시나리오: 팀장 승인 1회로 완료 ──
print("=== 소액 구매 (50만원) ===")
config_small = {"configurable": {"thread_id": "purchase-small"}}
for e in purchase_graph.stream({"item": "노트북", "amount": 500_000}, config_small):
    if "__interrupt__" in e:
        print(f"[중단] {e['__interrupt__'][0].value['message']}")

# TODO [Command 핵심]: Command(resume=True)로 팀장 승인 → process까지 한 번에 완료
for e in purchase_graph.stream(Command(resume=True), config_small):
    if "process" in e:
        print(f"→ {e['process']['status']}")

print("\n=== 고액 구매 (150만원) — interrupt 2회 ===")
config_large = {"configurable": {"thread_id": "purchase-large"}}
for e in purchase_graph.stream({"item": "서버", "amount": 1_500_000}, config_large):
    if "__interrupt__" in e:
        print(f"[중단1] {e['__interrupt__'][0].value['message']}")

# TODO [개념 확인]: 팀장 승인 후에도 바로 완료되지 않고 임원 interrupt가 또 발생한다.
#   → 같은 thread_id에서 interrupt가 순서대로 처리됨을 확인하세요.
for e in purchase_graph.stream(Command(resume=True), config_large):   # 팀장 승인
    if "__interrupt__" in e:
        print(f"[중단2] {e['__interrupt__'][0].value['message']}")
for e in purchase_graph.stream(Command(resume=True), config_large):   # 임원 승인
    if "process" in e:
        print(f"→ {e['process']['status']}")

### 5. Time Travel — 과거 체크포인트로 재실행

**Time Travel** 은 `get_state_history()`로 과거 체크포인트를 조회하고, 특정 시점의 `config`(= `checkpoint_id` 포함)로 그래프를 다시 실행하는 기능입니다.

```
히스토리 조회 (최신 → 과거 순):
  [0] step=4  next=()           ← 최신 (실행 완료)
  [1] step=3  next=("node_C",)
  [2] step=2  next=("node_B",)  ← 이 시점으로 되돌리기
  [3] step=1  next=("node_A",)

past_config = history[2].config  ← checkpoint_id 포함
graph.invoke(None, past_config)  → node_B, node_C 재실행
```

> **`update_state(past_config, values)`**: 재실행 전에 해당 체크포인트의 상태를 수정할 수 있다.

In [ ]:
# Time Travel 예제: 숫자를 단계적으로 변환하는 파이프라인

class TransformState(TypedDict):
    value: int
    log: list

def double(state: TransformState) -> dict:
    """값을 2배로 만든다."""
    v = state["value"] * 2
    print(f"  [double] {state['value']} → {v}")
    return {"value": v, "log": state["log"] + [f"double→{v}"]}

def add_ten(state: TransformState) -> dict:
    """10을 더한다."""
    v = state["value"] + 10
    print(f"  [add_ten] {state['value']} → {v}")
    return {"value": v, "log": state["log"] + [f"add_ten→{v}"]}

def square(state: TransformState) -> dict:
    """제곱한다."""
    v = state["value"] ** 2
    print(f"  [square] {state['value']} → {v}")
    return {"value": v, "log": state["log"] + [f"square→{v}"]}

transform_builder = StateGraph(TransformState)
transform_builder.add_node("double", double)
transform_builder.add_node("add_ten", add_ten)
transform_builder.add_node("square", square)
transform_builder.add_edge(START, "double")
transform_builder.add_edge("double", "add_ten")
transform_builder.add_edge("add_ten", "square")
transform_builder.add_edge("square", END)

transform_graph = transform_builder.compile(checkpointer=InMemorySaver())

In [ ]:
# 첫 실행: 5 → double(10) → add_ten(20) → square(400)
config = {"configurable": {"thread_id": "transform-001"}}

print("=== 첫 실행 ===")
result = transform_graph.invoke({"value": 5, "log": []}, config)
print(f"→ 최종값: {result['value']}, log: {result['log']}")

print("\n=== 체크포인트 히스토리 ===")
history = list(transform_graph.get_state_history(config))
for i, s in enumerate(history):
    print(f"  [{i}] step={s.metadata.get('step'):>2}  value={str(s.values.get('value')):>4}  next={s.next}")

In [ ]:
# double 완료 직후(add_ten 실행 전) 시점으로 되돌려 재실행
# history는 최신→과거 순: [0]=완료, [1]=square전, [2]=add_ten전, [3]=double전, [4]=시작전
past_config = history[2].config  # add_ten 실행 전 (value=10)

print("=== Time Travel: double 완료 시점으로 재실행 ===")
print(f"→ 재개 시점: value={history[2].values.get('value')}, next={history[2].next}\n")

result2 = transform_graph.invoke(None, past_config)
print(f"\n→ 재실행 결과: value={result2['value']}, log={result2['log']}")
print("(double 로그가 없으면 체크포인트에서 로드된 것)")

In [ ]:
# update_state로 값을 수정한 뒤 재실행
# double 완료 시점의 value=10 을 value=100 으로 바꾸면?
config3 = {"configurable": {"thread_id": "transform-002"}}
transform_graph.invoke({"value": 5, "log": []}, config3)  # 사전 실행

history3 = list(transform_graph.get_state_history(config3))
past_config3 = history3[2].config  # add_ten 실행 전

# 체크포인트 상태 수정 후 재실행
transform_graph.update_state(past_config3, {"value": 100})

print("=== update_state 후 재실행 (value=10 → 100) ===")
result3 = transform_graph.invoke(None, past_config3)
print(f"→ 결과: value={result3['value']}, log={result3['log']}")

#### 🎯 과제 4: Time Travel + update_state 로 재계산

3단계 가격 계산 파이프라인을 구현하고, `update_state()`로 중간 값을 수정한 뒤 재실행하세요.

| 노드 | 동작 |
|------|------|
| `apply_discount` | `price * (1 - discount_rate)` |
| `add_tax` | `price * 1.1` |
| `round_price` | `round(price)` |

**실습 흐름:**
1. `price=10000, discount_rate=0.1` 로 첫 실행
2. `get_state_history()`에서 `apply_discount` 완료 직후 체크포인트 찾기
3. `update_state(past_config, {"price": 7000})` 로 값 수정
4. `invoke(None, past_config)` 로 `add_tax`부터 재실행 후 결과 비교

힌트: `history` 리스트에서 `next=("add_tax",)` 인 항목을 찾으세요.

In [ ]:
class PriceState(TypedDict):
    price: float
    discount_rate: float

def apply_discount(state: PriceState) -> dict:
    """할인율을 적용한다."""
    discounted = state["price"] * (1 - state["discount_rate"])
    print(f"  [apply_discount] {state['price']} → {discounted}")
    return {"price": discounted}

def add_tax(state: PriceState) -> dict:
    """10% 세금을 추가한다."""
    taxed = state["price"] * 1.1
    print(f"  [add_tax] {state['price']} → {taxed}")
    return {"price": taxed}

def round_price(state: PriceState) -> dict:
    """가격을 정수로 반올림한다."""
    rounded = round(state["price"])
    print(f"  [round_price] {state['price']} → {rounded}")
    return {"price": rounded}

price_builder = StateGraph(PriceState)
price_builder.add_node("apply_discount", apply_discount)
price_builder.add_node("add_tax", add_tax)
price_builder.add_node("round_price", round_price)
price_builder.add_edge(START, "apply_discount")
price_builder.add_edge("apply_discount", "add_tax")
price_builder.add_edge("add_tax", "round_price")
price_builder.add_edge("round_price", END)
price_graph = price_builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "price-001"}}
result = price_graph.invoke({"price": 10000, "discount_rate": 0.1}, config)
print(f"\n첫 실행 결과: {result['price']}")  # 10000 * 0.9 * 1.1 = 9900

# TODO [Time Travel 핵심]: get_state_history()에서 next=("add_tax",) 인 항목을 찾아야 한다.
#   이 항목이 apply_discount 완료 직후 = add_tax 실행 직전 체크포인트이다.
history = list(price_graph.get_state_history(config))
print("\n=== 히스토리 ===")
for i, s in enumerate(history):
    print(f"  [{i}] next={s.next}  price={s.values.get('price')}")

# TODO [past_config 선택]: next=("add_tax",) 인 항목의 .config 를 가져오세요.
past_config = next(s.config for s in history if s.next == ("add_tax",))
print(f"\n재개 시점 price={price_graph.get_state(past_config).values['price']}")

# TODO [update_state 핵심]: 체크포인트의 price를 7000으로 수정한 뒤 재실행한다.
#   update_state는 해당 체크포인트 상태를 덮어쓴다 — invoke 전에 호출해야 한다.
price_graph.update_state(past_config, {"price": 7000.0})

print("\n=== update_state 후 재실행 (price=9000→7000으로 수정) ===")
result2 = price_graph.invoke(None, past_config)
print(f"재실행 결과: {result2['price']}")  # 7000 * 1.1 = 7700

# TODO [개념 확인]: 두 결과를 비교하세요.
#   첫 실행:  10000 * 0.9 = 9000 → * 1.1 = 9900 → round = 9900
#   재실행:   (수정됨) 7000      → * 1.1 = 7700 → round = 7700
print(f"\n비교: 첫 실행={result['price']}  /  update_state 후={result2['price']}")

### 6. 종합 과제 — 문서 처리 파이프라인 구축

지금까지 배운 모든 개념을 통합하여 **AI 문서 처리 파이프라인** 을 구축합니다.

**요구사항:**

| 요구사항 | 사용 기술 |
|---------|----------|
| 실행 히스토리 조회 | `compile(checkpointer=)`, `get_state_history()` |
| `analyze` 노드 첫 번째 시도 실패 → 재시도 성공 | 결함 복구 (`invoke(None, config)`) |
| 발행 전 사람의 최종 승인 | `interrupt()` + `Command(resume=...)` |
| 거절 시 `extract` 이후 체크포인트로 돌아가 재실행 | Time Travel + `invoke(None, past_config)` |

**노드 구성:**
- `extract(state)` — 문서에서 키워드 추출 (`keywords: list`)
- `analyze(state)` — 키워드 분석 및 요약 생성 (`summary: str`) — **첫 번째 호출은 실패**
- `review(state)` — `interrupt()`로 사람 검토 요청 (`feedback: str`)
- `publish(state)` — 승인 시 발행, 거절 시 `status="거절"` 기록 (`status: str`)

In [ ]:
# ── 종합 과제: 문서 처리 파이프라인 ──

class DocState(TypedDict):
    document: str
    keywords: list
    summary: str
    feedback: str
    status: str

analyze_fail_count = {"count": 0}

def extract(state: DocState) -> dict:
    """문서에서 키워드를 추출한다."""
    keywords = state["document"].split()
    print(f"  [extract] keywords={keywords}")
    return {"keywords": keywords}

def analyze(state: DocState) -> dict:
    """키워드를 분석하고 요약을 생성한다 — 첫 번째 호출 실패."""
    analyze_fail_count["count"] += 1
    print(f"  [analyze] 시도 {analyze_fail_count['count']}회")
    # TODO [결함 복구]: count==1이면 RuntimeError를 발생시킨다.
    #   이 예외로 그래프가 중단되고, invoke(None, config)로 재개할 때
    #   extract는 재실행되지 않고 analyze만 재시도된다.
    if analyze_fail_count["count"] == 1:
        raise RuntimeError("분석 서버 오류 (시뮬레이션)")
    return {"summary": f"키워드 {state['keywords']} 기반 요약 완료"}

def review(state: DocState) -> dict:
    """사람이 요약을 검토하고 피드백을 제공한다."""
    # TODO [interrupt]: interrupt()로 summary를 전달하고 피드백을 반환받는다.
    #   반환값을 feedback 키에 저장한다.
    feedback = interrupt({"message": "요약을 검토하세요.", "summary": state["summary"]})
    return {"feedback": feedback}

def publish(state: DocState) -> dict:
    """피드백에 따라 발행하거나 거절 처리한다."""
    if state["feedback"] == "승인":
        status = "발행 완료"
    else:
        status = f"거절: {state['feedback']}"
    print(f"  [publish] {status}")
    return {"status": status}

# TODO [체크포인팅]: compile 시 반드시 checkpointer를 지정해야
#   결함 복구와 Time Travel 모두 동작한다.
doc_builder = StateGraph(DocState)
doc_builder.add_node("extract", extract)
doc_builder.add_node("analyze", analyze)
doc_builder.add_node("review", review)
doc_builder.add_node("publish", publish)
doc_builder.add_edge(START, "extract")
doc_builder.add_edge("extract", "analyze")
doc_builder.add_edge("analyze", "review")
doc_builder.add_edge("review", "publish")
doc_builder.add_edge("publish", END)
doc_graph = doc_builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "doc-001"}}
initial = {"document": "LangGraph durable execution 학습"}

# ── 시나리오 1: 결함 복구 + 승인 ──
print("=== [1단계] 첫 실행 — analyze 실패 예상 ===")
try:
    doc_graph.invoke(initial, config)
except Exception as e:
    print(f"→ 예외: {e}")
    # TODO [결함 복구 확인]: get_state(config).next 를 출력하여
    #   어느 노드에서 멈췄는지 확인하세요. ("analyze",) 이어야 한다.
    print(f"→ 멈춘 위치: {doc_graph.get_state(config).next}")

print("\n=== [2단계] invoke(None, config) — analyze 재시도, extract 건너뜀 ===")
for event in doc_graph.stream(None, config):
    if "__interrupt__" in event:
        data = event["__interrupt__"][0].value
        print(f"→ [중단] {data['message']}")
        print(f"→ 요약: {data['summary']}")

print("\n=== [3단계] 승인 후 발행 ===")
for event in doc_graph.stream(Command(resume="승인"), config):
    if "publish" in event:
        print(f"→ {event['publish']['status']}")

# ── 시나리오 2: 거절 후 Time Travel 재작성 ──
print("\n" + "="*60)
print("=== 시나리오 2: 거절 후 Time Travel ===")

analyze_fail_count["count"] = 0  # 카운터 초기화
config2 = {"configurable": {"thread_id": "doc-002"}}

try:
    doc_graph.invoke(initial, config2)
except Exception:
    pass
for event in doc_graph.stream(None, config2):
    if "__interrupt__" in event:
        print("→ [중단] 검토 대기 중...")

# 거절
for event in doc_graph.stream(Command(resume="내용 부족"), config2):
    if "publish" in event:
        print(f"→ 1차 결과: {event['publish']['status']}")

# TODO [Time Travel]: get_state_history()에서 next=("analyze",) 인 체크포인트를 찾는다.
#   이 시점 = extract 완료 직후. 여기서 재실행하면 analyze부터 다시 시작된다.
history2 = list(doc_graph.get_state_history(config2))
past_config2 = next(s.config for s in history2 if s.next == ("analyze",))
print(f"\n→ Time Travel 시점: next={doc_graph.get_state(past_config2).next}")

# TODO [개념 확인]: analyze_fail_count를 초기화하지 않으면 어떻게 되나요?
analyze_fail_count["count"] = 0  # 재실행 전 카운터 초기화 필수

for event in doc_graph.stream(None, past_config2):
    if "__interrupt__" in event:
        print("→ [중단] 재검토 대기 중...")

for event in doc_graph.stream(Command(resume="승인"), past_config2):
    if "publish" in event:
        print(f"→ 재작성 후 결과: {event['publish']['status']}")